## 🔐 Fine-tuning BERT for Phishing URL Classification

In this notebook, we will **fine-tune** BERT (~110M parameters) to classify URLs as **Safe** or **Phishing**.
We'll go through the **full practical pipeline**: data → tokenization → training → evaluation → saving the model.

---

### 🧠 What is fine-tuning?

**Fine-tuning** means taking a model already pretrained on massive text (Wikipedia + Books) and **adapting it to your specific task** by training a small classification head and slightly adjusting the internal weights.

During pretraining, BERT learned language using two objectives:

* **Masked Language Modeling (MLM)** — predict missing words from both sides of context

  > “The weather is [MASK] today.” → *nice*

* **Next Sentence Prediction (NSP)** — understand relationships between sentences

Because of this, BERT already understands **syntax, context, and sentence structure**.
Fine-tuning teaches it how to apply that knowledge to **your labels** (Safe vs Phishing).

---

### 🧩 Why BERT is a great fit for URL classification

Although URLs are not natural sentences, they contain **meaningful patterns**:

* words: `login`, `secure`, `verify`, `account`
* structure: subdomains, paths, tokens, separators
* obfuscation tricks common in phishing

BERT’s **subword tokenization** and contextual understanding help it learn these patterns effectively.

---

### 🤔 Why fine-tune instead of calling an API LLM?

Fine-tuning a local BERT model is:

* ✅ **Free at inference time**
* ✅ **Private** (no data leaves your environment)
* ✅ **Fast** (milliseconds per prediction)
* ✅ **Easy to deploy** in APIs/backends
* ✅ **Fully controllable** (you own the model and data)

Use this approach when you need **privacy, speed, and independence** from external services.

---

### 🛠️ Best practices for BERT fine-tuning (classification)

* Use `AutoModelForSequenceClassification`
* Keep `label2id` / `id2label` consistent
* Use `DataCollatorWithPadding` for dynamic batching
* Track validation metrics and apply early stopping
* Save **model + tokenizer + labels** together for reuse

---

👉 Ready? Let’s install the libraries and start building the pipeline step by step.


In [1]:
# !pip install evaluate --quiet

In [2]:
# !pip show evaluate

## 📦 Imports & Setup

We’ll use three core libraries from the Hugging Face ecosystem:

* datasets → ready-to-use NLP datasets
* transformers → pretrained BERT + training tools
* evaluate → standard metrics (accuracy, f1, etc.)


In [3]:
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

/opt/miniconda3/envs/mlpeople7/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



---

## 📚 Loading a Dataset

Instead of preparing data manually, we can pull a phishing URL dataset directly from Hugging Face.

For this tutorial, use:

**`shawhin/phishing-site-classification`**

This dataset already contains:

* `url` — the input text
* `label` — 0 (safe) or 1 (phishing)
* predefined train / test splits

Perfect for practicing BERT fine-tuning without extra preprocessing.


In [4]:
dataset_dict = load_dataset(
    "shawhin/phishing-site-classification"
)

In [5]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
})

## 🤖 Load the Model and Tokenizer

At this stage, we load a pretrained BERT and the matching tokenizer from
**Transformers**.

* `AutoTokenizer.from_pretrained(...)` → automatically picks the correct tokenizer
* `AutoModelForSequenceClassification.from_pretrained(...)` → loads BERT **with a classification head on top**

We explicitly define:

* `num_labels=2` → **Safe** vs **Phishing**
* `id2label` / `label2id` → lets the model map numbers ↔ human-readable classes during training and inference (useful for logs, metrics, and predictions)

👉 Under the hood, this is **BERT + a small linear layer** that converts the 768-dim embedding into **2 class logits**.



In [6]:
model_path = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_path)

label2id = {"safe": 0, "phishing": 1}
id2label = {0: "safe", 1: "phishing"}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

/opt/miniconda3/envs/mlpeople7/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### ⚠️ About the warning when loading `BertForSequenceClassification`

You may see:

> *Some weights of BertForSequenceClassification were not initialized from the model checkpoint at `google-bert/bert-base-uncased` and are newly initialized: `['classifier.bias', 'classifier.weight']`*

This means:

> **The classification head was created from scratch and was not loaded from the pretrained checkpoint.**

**🔍 Why this happens**

* `bert-base-uncased` is a pretrained **language model** (MLM + NSP), not a classifier.
* `BertForSequenceClassification` adds an extra **linear layer (`classifier`)** on top of BERT.
* That layer **does not exist** in the original checkpoint, so **Hugging Face** initializes it with random weights.

**✅ Why this is expected (and correct)**

This is exactly how fine-tuning works:

1. Load pretrained BERT encoder (already understands language)
2. Add a small classification head
3. Train this head (and slightly adjust BERT) on your task

💡 Until you train the model, the classifier is random - so predictions will be meaningless.



---

### ❓ Can `bert-base-uncased` be used “as is” for classification?

**No — not directly.**

BERT base uncased is pretrained for **language understanding** (MLM + NSP), **not** for classifying text into labels like *“Safe”* / *“Not Safe”*.

You have two options:

1. **Fine-tune BERT on your dataset** → ✅ the standard and recommended approach.
2. **Use zero-shot classification** with a model already trained for entailment-style labeling.

Example:

```python
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

result = classifier(
    "your input",
    candidate_labels=["Safe", "Not Safe"]
)

print(result)
```

This uses BART large MNLI, which was trained on Natural Language Inference and can generalize to unseen labels.

> 💡 Note: this is **not BERT**.
> It’s a model specifically trained to decide whether a sentence *entails* a label — which is why zero-shot works.


In [7]:
import torch

# model.eval() won't help - classificator not trained
model.eval()
inputs = tokenizer("test text", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
print(logits)

tensor([[ 0.3954, -0.0323]])


We got random logits because classifier.weight and bias are not yet trained.

## 🧊 Preparing the model (Freezing most layers)

Before training, we **freeze the BERT encoder** and allow only the **final classification layers** to learn.

With BERT base uncased, the flow is:

```
tokens → BERT encoder → [CLS] embedding → classifier → label
```

We keep the encoder weights fixed and train only the **classifier head** (optionally the pooler). This turns BERT into a strong **feature extractor** while a small head learns your task.

**Why do this?**

* Faster training
* Works well for small datasets
* Reduces overfitting
* More stable optimization

Optionally, you can **unfreeze more layers** for full fine-tuning if needed.


In [8]:
# Show first and last 10 parameter entries
params = list(model.named_parameters())
layer_count = len(params)

for i, (name, param) in enumerate(params):
    if i < 10 or i >= layer_count - 10:
        print(f"{i:4d} | {name:60s} | requires_grad={param.requires_grad}")
    elif i == 10:
        print(" ...")

   0 | bert.embeddings.word_embeddings.weight                       | requires_grad=True
   1 | bert.embeddings.position_embeddings.weight                   | requires_grad=True
   2 | bert.embeddings.token_type_embeddings.weight                 | requires_grad=True
   3 | bert.embeddings.LayerNorm.weight                             | requires_grad=True
   4 | bert.embeddings.LayerNorm.bias                               | requires_grad=True
   5 | bert.encoder.layer.0.attention.self.query.weight             | requires_grad=True
   6 | bert.encoder.layer.0.attention.self.query.bias               | requires_grad=True
   7 | bert.encoder.layer.0.attention.self.key.weight               | requires_grad=True
   8 | bert.encoder.layer.0.attention.self.key.bias                 | requires_grad=True
   9 | bert.encoder.layer.0.attention.self.value.weight             | requires_grad=True
 ...
 191 | bert.encoder.layer.11.intermediate.dense.weight              | requires_grad=True
 192 | bert.enco

In [9]:
# Freeze whole BERT
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze pooling layers
for param in model.bert.pooler.parameters():
    param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

In [10]:
for i, (name, param) in enumerate(params):
    if i < 10 or i >= layer_count - 10:
        print(f"{i:4d} | {name:60s} | requires_grad={param.requires_grad}")
    elif i == 10:
        print(" ...")

   0 | bert.embeddings.word_embeddings.weight                       | requires_grad=False
   1 | bert.embeddings.position_embeddings.weight                   | requires_grad=False
   2 | bert.embeddings.token_type_embeddings.weight                 | requires_grad=False
   3 | bert.embeddings.LayerNorm.weight                             | requires_grad=False
   4 | bert.embeddings.LayerNorm.bias                               | requires_grad=False
   5 | bert.encoder.layer.0.attention.self.query.weight             | requires_grad=False
   6 | bert.encoder.layer.0.attention.self.query.bias               | requires_grad=False
   7 | bert.encoder.layer.0.attention.self.key.weight               | requires_grad=False
   8 | bert.encoder.layer.0.attention.self.key.bias                 | requires_grad=False
   9 | bert.encoder.layer.0.attention.self.value.weight             | requires_grad=False
 ...
 191 | bert.encoder.layer.11.intermediate.dense.weight              | requires_grad=False
 192 

## 🧾 Text Preprocessing

Before training, we need to convert raw text into token IDs that BERT can understand using BERT base uncased tokenizer.


### 🔤 Tokenization function

We define a preprocessing function that:

In [11]:
# define text preprecessing
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

### 📦 Apply to the full dataset

We apply tokenization to the entire dataset using batch processing (faster and more efficient):


In [12]:
# tokenize full dataset
tokenized_data = dataset_dict.map(preprocess_function, batched=True)

Map: 100%|██████████| 450/450 [00:00<00:00, 11327.93 examples/s]



---

### 📏 Dynamic Padding with Data Collator

Texts in a batch often have **different lengths**, so we cannot stack them directly into tensors.

To solve this, we use:

`DataCollatorWithPadding`

---

**🧠 What it does**

* Finds the longest sequence in each batch
* Pads all other sequences to the same length
* Creates properly shaped tensors for training

---

**⚙️ Why it is important**

Without dynamic padding:

* you would waste memory (fixed max-length padding everywhere)
* or get shape mismatch errors during batching

With it:

* padding is applied **only when needed per batch**
* training becomes more efficient and flexible


In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 📊 Evaluation Metrics

To evaluate our fine-tuned BERT base uncased classifier, we use standard classification metrics from the Evaluate library.

We measure:

* **Accuracy** → how many predictions are correct
* **ROC AUC** → how well the model separates *Safe vs Phishing*


In [ ]:
# Load metrics
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

### 🧠 Metric function for Trainer

This function is called automatically during evaluation.

It:

* converts model logits → probabilities
* extracts positive class scores (phishing = 1)
* computes AUC
* computes accuracy

👉 AUC is especially important for **imbalanced datasets**, which is common in phishing detection.

In [ ]:
def compute_metrics(eval_pred):
    # Get predictions and real labels
    predictions, labels = eval_pred

    # apply softmax to obtain probabilities
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)

    # take the positive class probabilities to calculate the AUC
    positive_class_probs = probabilities[:, 1]

    # calculate AUC
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'], 3)

    # get the predicted classes (most likely)
    predicted_classes = np.argmax(predictions, axis=1)

    # calculate accuracy
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'], 3)

    return {"Accuracy": acc, "AUC": auc}

## 🏋️ Train the Model

Now we define the training configuration and fine-tune our BERT base uncased using the Hugging Face training pipeline.

We use the **Trainer API** from Transformers, which handles:

* training loop
* evaluation
* logging
* checkpointing


### ⚙️ Hyperparameters

* **learning_rate** → how fast the model updates weights
* **batch_size** → number of samples per step
* **epochs** → how many times the model sees the dataset

In [ ]:
# hyperparameters
lr = 2e-4
batch_size = 8
num_epochs = 10

### 🧠 Training configuration

👉 This setup ensures:

* evaluation after each epoch
* best model automatically saved
* training logs tracked per epoch

In [ ]:
training_args = TrainingArguments(
    output_dir="bert-phishing-classifier",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

### 🏁 Start training

#### ⚠️ Important note (performance)

Training BERT base uncased on CPU is **very slow**, because:

* ~110 million parameters
* transformer attention is computationally expensive

👉 Recommended setup:

* Google Colab GPU (T4 / A100)
* or local CUDA-enabled GPU

In [19]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Use GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not availabel. Use CPU.")

CUDA not availabel. Use CPU.


In [ ]:
%%time
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

                                                 
  3%|▎         | 83/2630 [01:56<05:12,  8.15it/s] 

{'loss': 0.4354, 'grad_norm': 1.2840896844863892, 'learning_rate': 0.00018, 'epoch': 1.0}
































                                                 
                                               

  3%|▎         | 83/2630 [02:03<05:12,  8.15it/s]



{'eval_loss': 0.38988539576530457, 'eval_Accuracy': 0.82, 'eval_AUC': 0.928, 'eval_runtime': 7.2873, 'eval_samples_per_second': 61.751, 'eval_steps_per_second': 7.822, 'epoch': 1.0}


                                                 
  3%|▎         | 83/2630 [02:41<05:12,  8.15it/s]   

...

                                                 
  3%|▎         | 83/2630 [03:26<05:12,  8.15it/s] 

{'loss': 0.3787, 'grad_norm': 0.8428274989128113, 'learning_rate': 0.00014, 'epoch': 3.0}































                                                 
                                               

  3%|▎         | 83/2630 [03:33<05:12,  8.15it/s]



{'eval_loss': 0.3159913122653961, 'eval_Accuracy': 0.88, 'eval_AUC': 0.939, 'eval_runtime': 6.8392, 'eval_samples_per_second': 65.797, 'eval_steps_per_second': 8.334, 'epoch': 3.0}


KeyboardInterrupt: 